### PepCL quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntranoslab/PepCL/blob/main/notebooks/01_pepcl_quickstart.ipynb)

This notebooks outlines how to get started with two primary workflows:

1. Standard PepCL updating of MHCPrime.
2. Memory-efficient PepCL microbatch updating (PepCL_Micro) of MHCPrime. This option is for lower compute.

Please download the MS and yeast display ([Holec et al., 2025](https://www.pnas.org/doi/10.1073/pnas.2514741122)) train and test sets from [here](https://drive.google.com/drive/folders/128SrLjjlG_L_KzFFxBwU_zmqtQyG4yow?usp=sharing) and follow the instructions below. Place datasets under `pepcl_data/`.

#### Setup notebook

In [ ]:
# notebook/repository setup
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)
print("Python executable:", sys.executable)

if IN_COLAB:
    os.chdir("/content")

    if not Path("PepCL").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/ntranoslab/PepCL.git"],
            check=True,
        )
    else:
        os.chdir("/content/PepCL")
        subprocess.run(["git", "pull"], check=True)
        os.chdir("/content")

    os.chdir("/content/PepCL")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "peptides",
            "transformers>=4.30,<4.58",
            "git+https://github.com/ntranoslab/MHCPrime.git@main",
            "--no-deps",
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-e",
            ".",
            "--no-deps",
        ],
        check=True,
    )

    REPO_ROOT = Path.cwd()

else:
    NOTEBOOK_DIR = Path.cwd()

    if NOTEBOOK_DIR.name == "notebooks":
        REPO_ROOT = NOTEBOOK_DIR.parent
    else:
        REPO_ROOT = NOTEBOOK_DIR

    os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Repository root:", REPO_ROOT)
print("src path added:", src_path)
print("src in sys.path:", src_path in sys.path)

In [ ]:
# colab only pepcl data download
# run this after REPO_ROOT is defined and before checking required data files.

if IN_COLAB:
    DATA_DIR = REPO_ROOT / "pepcl_data"

    required_data_files = [
        DATA_DIR / "ms_train_test_sets" / "ms_train_data.csv.gz",
        DATA_DIR / "ms_train_test_sets" / "ms_test_data.csv.gz",
        DATA_DIR / "holec_train_test_sets" / "holec_train_set.csv.gz",
        DATA_DIR / "holec_train_test_sets" / "holec_test_set.csv.gz",
    ]

    data_already_present = all(path.exists() for path in required_data_files)

    if data_already_present:
        print("PepCL data already found; skipping download.")
    else:
        print("PepCL data not found. Downloading from Google Drive...")

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "gdown"],
            check=True,
        )

        import gdown

        DATA_DIR.mkdir(parents=True, exist_ok=True)

        gdrive_folder_url = "https://drive.google.com/drive/folders/1xG_c22fytWbWLSlF_SjaYuC9Aq0DYDdG?usp=drive_link"

        downloaded_files = gdown.download_folder(
            url=gdrive_folder_url,
            output=str(DATA_DIR),
            quiet=False,
            use_cookies=False,
        )

        print("Downloaded files:")
        for path in downloaded_files or []:
            print(" ", path)
else:
    print("Not running in Colab; skipping Google Drive data download.")

In [ ]:
# Project paths
DATA_DIR = REPO_ROOT / "pepcl_data"
OUTPUT_DIR = REPO_ROOT / "pepcl_outputs"
CHECKPOINT_DIR = REPO_ROOT / "model_checkpoints"
CACHE_DIR = OUTPUT_DIR / "cache"

MS_DATA_DIR = DATA_DIR / "ms_train_test_sets"
HOLEC_DATA_DIR = DATA_DIR / "holec_train_test_sets"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MS_DATA_DIR:", MS_DATA_DIR)
print("HOLEC_DATA_DIR:", HOLEC_DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("CACHE_DIR:", CACHE_DIR)

# check that required data files exist
required_data_files = {
    "MS train": MS_DATA_DIR / "ms_train_data.csv.gz",
    "MS test": MS_DATA_DIR / "ms_test_data.csv.gz",
    "Holec train": HOLEC_DATA_DIR / "holec_train_set.csv.gz",
    "Holec test": HOLEC_DATA_DIR / "holec_test_set.csv.gz",
}

missing_files = []

for name, path in required_data_files.items():
    exists = path.exists()
    print(f"{name:12s}: {path} | exists={exists}")
    if not exists:
        missing_files.append((name, path))

if missing_files:
    raise FileNotFoundError(
        "Missing PepCL data files. Download the PepCL data bundle and place it under "
        f"{DATA_DIR} with the expected folder structure shown above."
    )

In [ ]:
# general imports
import gc
import pickle
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import copy

# warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# check versions
import mhcprime
import pepcl

print("mhcprime version:", mhcprime.__version__)
print("pepcl version:", pepcl.__version__)
print("torch version:", torch.__version__)

# mhcprime imports
from mhcprime import (
    load_mhcprime_model,
    load_default_mhc_pseudosequences,
    print_data_stats,
    predict_dataframe,
)

from mhcprime.checkpointing import load_state_dict_flexible
from mhcprime.model_utils import collate_fn
from mhcprime.inference import predict_with_models

# pepcl imports
from pepcl import (
    PepCL,
    PepCL_Micro,
    prepare_pepcl_training_dataframe,
    clear_memory,
    get_device,
    print_device_info,
    resolve_csv_path,
    set_seed,
    get_default_pepcl_params,
    get_default_pepcl_micro_params,
)

from pepcl.plotting import plot_training_losses, plot_bars_or_boxplots, plot_bars_or_boxplots_grid
from pepcl.metrics import compute_metric_matrix

from pepcl.assets import (
    get_default_data_dir,
    get_default_output_dir,
    get_default_checkpoint_dir,
    require_path,
)

In [ ]:
# reproducibility
SEED = 42
set_seed(SEED)

DEVICE = print_device_info()

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print(
        "No CUDA GPU detected. In Colab, go to Runtime → Change runtime type → "
        "Hardware accelerator → GPU, then restart and rerun."
    )

#### Import and process datasets for PepCL

 - Import the MS and yeast display datasets here for initial testing.
 - The current workflow matches the MS train/replay data to the allele-length distribution of the yeast display dataset (not the counts, but the unique group types). However, alternative configurations can be enabled with the processing function shown below.
 - Both test sets are held at a 1:99 positive to negative ratio and contain no overlaps with either train sets at a peptide-allele level.
 - Get the data described above and place in `pepcl_data`:
    ```text
    PepCL/
    ├── pepcl_data/
    │   ├── ms_train_test_sets/
    │   │   ├── ms_train_data.csv.gz
    │   │   └── ms_test_data.csv.gz
    │   └── holec_train_test_sets/
    │       ├── holec_train_set.csv.gz
    │       └── holec_test_set.csv.gz
    └── notebooks/
        └── 01_pepcl_quickstart.ipynb
    ```

In [ ]:
%%time
# get datasets

ms_train_data = pd.read_csv(MS_DATA_DIR / "ms_train_data.csv.gz", index_col=0)
ms_test_data = pd.read_csv(MS_DATA_DIR / "ms_test_data.csv.gz", index_col=0)

holec_train_set = pd.read_csv(HOLEC_DATA_DIR / "holec_train_set.csv.gz", index_col=0)
holec_test_set = pd.read_csv(HOLEC_DATA_DIR / "holec_test_set.csv.gz", index_col=0)

print("MS train dataset")
print_data_stats(ms_train_data)
print(end="\n")

print("MS test dataset")
print_data_stats(ms_test_data)
print(end="\n")

print("Holec train dataset")
print_data_stats(holec_train_set)
print(end="\n")

print("Holec test dataset")
print_data_stats(holec_test_set)
print(end="\n")

In [ ]:
%%time
# merge and process data
PEPCL_PROCESSING_CONFIG = {
    "match_ms_to_new_allele_length": True,
    "drop_new_unmatched_allele_length": False,
    "process": True,
    "create_seq_allele": True,
    "remove_flank": True,
    "add_pseudo_label": False,
    "print_stats": True,
    "verbose": True,
}

ms_holec_train_set = prepare_pepcl_training_dataframe(
    ms_train_df=ms_train_data,
    new_train_df=holec_train_set,
    **PEPCL_PROCESSING_CONFIG,
)

print("\nFinal merged PepCL training dataframe:")
print_data_stats(ms_holec_train_set)

display(ms_holec_train_set.head())

In [ ]:
# get allele and length matched test data
ms_test_data["seq_len"]=ms_test_data.seq.str.replace("$", "").apply(len)
ms_test_data_allele_length_matched=ms_test_data[ms_test_data.allele.isin(holec_test_set.allele.unique().tolist())].query("seq_len == 9").copy()
print_data_stats(ms_test_data_allele_length_matched)

#### Load MHCPrime

In [ ]:
# load mhcprime
mhcprime_base, tokenizer, model_params = load_mhcprime_model(
    checkpoint_path=None,   # use packaged MHCPrime base checkpoint
    device=DEVICE,
    strict=True,
    eval_mode=True,
    print_params=True,
    print_check=False,
)

print("MHCPrime base model loaded.")
print("Model device:", next(mhcprime_base.parameters()).device)

clear_memory()

#### Run PepCL

 - Assumes adequate compute, otherwise, run PepCL_Micro in the section below.
 - See the base parameters below. These are currently set to those used in the main manuscript. Please see the documentation for editing and customizing these settings.
 - PepCL supports both conventional fine-tuning (FT) and replay-based updating with drift regularization. Both are implemented below.
 - If running in Colab, potentially reduce the `num_workers`.

In [ ]:
# get default params
pepcl_base_params = get_default_pepcl_params(
    tokenizer=tokenizer,
    device=str(DEVICE),
    collate_fn=collate_fn,
    seed=SEED,
)

In [ ]:
# init models
holec_ft = copy.deepcopy(mhcprime_base)
holec_pepcl = copy.deepcopy(mhcprime_base)

holec_ft.to(DEVICE)
holec_pepcl.to(DEVICE)

holec_ft.train()
holec_pepcl.train()

clear_memory()

In [ ]:
%%time
# run ft
print("Run FT (no replay or drift regularization)")
pepcl=PepCL(
            model=holec_ft,
            train_df=ms_holec_train_set,
            use_replay = False,
            **pepcl_base_params,
)
holec_ft_training_stats=pepcl.train()

In [ ]:
%%time
# run ft
print("Run PepCL (replay and drift regularization)")
pepcl=PepCL(
            model=holec_pepcl,
            train_df=ms_holec_train_set,
            use_replay = True,
            **pepcl_base_params,
)
holec_pepcl_training_stats=pepcl.train()

In [ ]:
# optional, save updated models

SAVE_UPDATED_MODELS = False
# SAVE_UPDATED_MODELS = True

HOLEC_MODEL_DIR = CHECKPOINT_DIR / "holec_models"
HOLEC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_UPDATED_MODELS:
    torch.save(holec_ft.state_dict(), HOLEC_MODEL_DIR / "holec_ft_model.pt")
    torch.save(holec_pepcl.state_dict(), HOLEC_MODEL_DIR / "holec_pepcl_model.pt")

    print("Saved updated model weights:")
    print("FT-only:", HOLEC_MODEL_DIR / "holec_ft_model.pt")
    print("PepCL:  ", HOLEC_MODEL_DIR / "holec_pepcl_model.pt")
else:
    print("SAVE_UPDATED_MODELS=False; skipping checkpoint save.")

In [ ]:
# plot losses
fig, axes = plot_training_losses(
    holec_ft_training_stats,
    holec_pepcl_training_stats,
    left_title="Fine-tuning loss",
    right_title="PepCL replay losses",
    left_xlabel="Step",
    left_ylabel="Loss",
    right_xlabel="Step",
    right_ylabel="Loss",
    ft_only_label="FT-only",
    pepcl_ft_label="PepCL",
    ms_label="MS replay",
    dr_label="Drift regularization",
    ft_only_color="#160D3B",
    pepcl_ft_color="#3985F8",
    ms_color="#160D3B",
    dr_color="#3985F8",
    line_width=1.5,
    spine_width=0.8,
    tick_length=4.0,
    tick_width=0.8,
    tick_labelsize=10,
    label_fontsize=11,
    title_fontsize=12,
    legend_fontsize=9,
    legend_y=-0.20,
    wspace=0.25,
    bottom_margin=0.26,
    figsize=(11, 4),
    dpi=300,
    default_font=None,
    save_path=None,
)

#### Run PepCL_Micro

 - This section runs a more memory-efficient PepCL for lower compute systems. If you only want to run standard PepCL and score the updated models, you can skip this section and jump directly to [Inference](#Inference).
 - PepCL_Micro exists as a branch within PepCL that uses microbatching to approximate the batchwise LogSmoothAP loss for an effective batch size of 3,072. Use when lower compute is available.

In [ ]:
# get default params
pepcl_micro_base_params = get_default_pepcl_micro_params(
    tokenizer=tokenizer,
    device=str(DEVICE),
    collate_fn=collate_fn,
    seed=SEED,
)

In [ ]:
# init models
holec_ft_micro = copy.deepcopy(mhcprime_base)
holec_pepcl_micro = copy.deepcopy(mhcprime_base)

holec_ft_micro.to(DEVICE)
holec_pepcl_micro.to(DEVICE)

holec_ft_micro.train()
holec_pepcl_micro.train()

clear_memory()

In [ ]:
%%time
# run ft
print("Run FT micro (no replay or drift regularization)")
pepcl=PepCL(
            model=holec_ft_micro,
            train_df=ms_holec_train_set,
            use_replay = False,
            **pepcl_micro_base_params, # make sure these are the micro params
)
holec_ft_micro_training_stats=pepcl.train()

In [ ]:
%%time
# run ft
print("Run PepCL micro (replay and drift regularization)")
pepcl=PepCL(
            model=holec_pepcl_micro,
            train_df=ms_holec_train_set,
            use_replay = True,
            **pepcl_micro_base_params, # make sure these are the micro params
)
holec_pepcl_micro_training_stats=pepcl.train()

In [ ]:
# optional, save updated models

SAVE_UPDATED_MODELS = False
# SAVE_UPDATED_MODELS = True

HOLEC_MODEL_DIR = CHECKPOINT_DIR / "holec_models"
HOLEC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_UPDATED_MODELS:
    torch.save(holec_ft_micro.state_dict(), HOLEC_MODEL_DIR / "holec_ft_micro_model.pt")
    torch.save(holec_pepcl_micro.state_dict(), HOLEC_MODEL_DIR / "holec_pepcl_micro_model.pt")

    print("Saved updated model weights:")
    print("FT-only:", HOLEC_MODEL_DIR / "holec_ft_micro_model.pt")
    print("PepCL:  ", HOLEC_MODEL_DIR / "holec_pepcl_micro_model.pt")
else:
    print("SAVE_UPDATED_MODELS=False; skipping checkpoint save.")

In [ ]:
# plot losses
fig, axes = plot_training_losses(
    holec_ft_micro_training_stats,
    holec_pepcl_micro_training_stats,
    left_title="Fine-tuning loss",
    right_title="PepCL replay losses",
    left_xlabel="Step",
    left_ylabel="Loss",
    right_xlabel="Step",
    right_ylabel="Loss",
    ft_only_label="FT-only",
    pepcl_ft_label="PepCL",
    ms_label="MS replay",
    dr_label="Drift regularization",
    ft_only_color="#160D3B",
    pepcl_ft_color="#3985F8",
    ms_color="#160D3B",
    dr_color="#3985F8",
    line_width=1.5,
    spine_width=0.8,
    tick_length=4.0,
    tick_width=0.8,
    tick_labelsize=10,
    label_fontsize=11,
    title_fontsize=12,
    legend_fontsize=9,
    legend_y=-0.20,
    wspace=0.25,
    bottom_margin=0.26,
    figsize=(11, 4),
    dpi=300,
    default_font=None,
    save_path=None,
)

#### Inference

 - Include the base MHCPrime model for inference if running for the first time. Comment out the micro models if PepCL_Micro was not run (or comment out the main PepCL models if those were not run).

In [ ]:
# get models to run
base_model = copy.deepcopy(mhcprime_base)

base_and_updated_models = {

    # base model
    "mhcprime": base_model,

    # pepcl updated models if run
    "holec_ft": holec_ft,
    "holec_pepcl": holec_pepcl,

    # pepcl_micro updated models if run
    "holec_ft_micro": holec_ft_micro,
    "holec_pepcl_micro": holec_pepcl_micro,
}

for model_name, model in base_and_updated_models.items():
    model.to(DEVICE)
    model.eval()

print("Models prepared for inference:")
for model_name in base_and_updated_models:
    print(" ", model_name)

In [ ]:
# Fast inference settings
INFERENCE_BATCH_SIZE = 3072
INFERENCE_NUM_WORKERS = 16

# For smaller local machines or Colab, reduce these if needed:
# INFERENCE_BATCH_SIZE = 512
# INFERENCE_NUM_WORKERS = 2

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
%%time
# run inference on ms test data (matched allele length)
ms_test_data_allele_length_matched = predict_with_models(
    models=base_and_updated_models,
    df=ms_test_data_allele_length_matched,
    tokenizer=tokenizer,
    mode="fast",
    batch_size=INFERENCE_BATCH_SIZE,
    num_workers=INFERENCE_NUM_WORKERS,
    device=DEVICE,
    preprocess=True,
    cache_dir=CACHE_DIR,
    cache_name="ms_test_cache.pt",
    remove_cache=True,
    add_rank=False,
    return_processed=False,
    disable_tqdm=False,
    use_bf16=True,
    compile_model=False,
    use_domain_id=False,
)

In [ ]:
%%time
# run inference on holec test data
holec_test_set = predict_with_models(
    models=base_and_updated_models,
    df=holec_test_set,
    tokenizer=tokenizer,
    mode="fast",
    batch_size=INFERENCE_BATCH_SIZE,
    num_workers=INFERENCE_NUM_WORKERS,
    device=DEVICE,
    preprocess=True,
    cache_dir=CACHE_DIR,
    cache_name="holec_test_cache.pt",
    remove_cache=True,
    add_rank=False,
    return_processed=False,
    disable_tqdm=False,
    use_bf16=True,
    compile_model=False,
    use_domain_id=False,
)

In [ ]:
# optional, save the output dataframes with scores

SAVE_SCORED_OUTPUTS = False
# SAVE_SCORED_OUTPUTS = True

if SAVE_SCORED_OUTPUTS:
    scored_dir = OUTPUT_DIR / "holec_scored_outputs"
    scored_dir.mkdir(parents=True, exist_ok=True)

    ms_test_data_allele_length_matched.to_csv(scored_dir / "ms_test_allele_length_matched_scored.csv.gz", index=False)
    holec_test_set.to_csv(scored_dir / "holec_test_scored.csv.gz", index=False)

    print("Saved scored outputs:")
    print(" MS test:", scored_dir / "ms_test_allele_length_matched_scored.csv.gz")
    print(" Holec test:", scored_dir / "holec_test_scored.csv.gz")
else:
    print("SAVE_SCORED_OUTPUTS=False; skipping output save.")

#### Benchmarking

 - Plot results for PepCL and PepCL_Micro (if run).

In [ ]:
# get score cols and compute metrics
score_cols=["mhcprime", ]
score_cols+=["holec_ft", "holec_pepcl"]
score_cols+=["holec_ft_micro", "holec_pepcl_micro"]

# methods: "ap", "auc", "auc01", "ppvn", "spearman"
ms_allele_ap_matrix=compute_metric_matrix(ms_test_data_allele_length_matched, score_cols, method="ap", disable_tqdm=False)

holec_allele_ap_matrix=compute_metric_matrix(holec_test_set, score_cols, method="ap", disable_tqdm=False)

In [ ]:
"""
Plotting parameters for boxplots.
"""
score_cols=["mhcprime", "holec_ft", "holec_pepcl", "holec_ft_micro", "holec_pepcl_micro"]

model_groups = {
    'MHCPrime': ["mhcprime"],
    'FT': ["holec_ft"],
    'PepCL': ["holec_pepcl"],
    'FT (micro)': ["holec_ft_micro"],
    'PepCL (micro)': ["holec_pepcl_micro"],
}

x_ticks = ['']

colors = {
    'MHCPrime': '#081470',
    'FT': '#bababa',
    'PepCL': '#9bb0ed',
    'FT (micro)': '#555656',
    'PepCL (micro)': '#2947a1',
}

plot_params = {
    'title': '',
    'title_y': 1.04,
    'title_bbox': None,
    'title_fontsize': 6,
    'title_bold': False,
    'xlabel': '',
    'ylabel': 'AP',
    'xlabel_fontsize': 6,
    'ylabel_fontsize': 6,
    'figsize': (1.80, 1.65),
    'dpi': 300,
    'legend_fontsize': 6,
    'legend_cols': 3,
    'legend_handle_size': 0.7,
    'axis_linewidth': 0.35,
    'tick_length': 2.2,
    'xlabel_pad': 1,
    'ylabel_pad': 1.5,
    'xtick_labelsize': 5.5,
    'ytick_labelsize': 6,
}

plotting_parameters = dict(
    model_col='model_name',
    score_col='metric',
    model_groups=model_groups,
    x_ticks=[''],
    colors=colors,
    plot_params=plot_params,
    ylim=(0.0, 1.0),
    grid_alpha=0,
    mode='box',
    bar_width=0.32,
    bar_spacing=0.08,
    group_spacing=0.10,
    x_margin=0.12,
    bar_outline_color='black',
    box_median_width=0.60,
    bar_outline_width=0.40,
    box_whisker_width=0.40,
    box_flier_size=0,
    legend_mode=None,
    xtick_rotation=0,
    pval_df=None,
    pval_fontsize=4,
    pval_linewidth=0.35,
    pval_color='black',
    pval_bar_offset=0.035,
    pval_bar_spacing=0.06,
    pval_text_offset=0.004,
    pval_star_text_offset=-0.01,
    pval_ns_text_offset=0.004,
)

# plot all together
plot_dict = {
    'yeast_display': {
        'title': 'Yeast Display AP',
        'df': holec_allele_ap_matrix,
    },
    'ms_for_yeast': {
        'title': 'MS AP',
        'df': ms_allele_ap_matrix,
    },
}

fig, axes = plot_bars_or_boxplots_grid(
    plot_dict=plot_dict,
    legend_dict=colors,
    legend_fontsize=6,
    legend_swatch_size=0.6,
    legend_outline_width=0.3,
    legend_height=0.5,
    panel_gap=0.0,
    grid_dpi=300,
    save_path=None,
    **plotting_parameters,
)
